# كشكول التجارب: تطبيق إعادة الترتيب عملياً باستخدام الـ LLM (المحاضرة 45)

أهلاً بيك في معمل المحاضرة 45!
في الفيديو اللي فات اتعلمنا نظرياً إيه هي فكرة الـ Reranking وليه بنحتاجها.
النهارده المحاضر طبق الفكرة دي بشكل عملي وذكي جداً:
هنستخدم النموذج اللغوي الكبير (LLM) نفسه كـ Reranker!
هنحمل ملف نصوص كامل عن لانج تشين (`langchain_sample.txt`)، هنقطعه ونخزنه في FAISS، وبعدها هنطلب من النموذج يرتب الوثائق المسترجعة بالأرقام من الأكثر صلة للأقل صلة بالسؤال!

### 1. تجهيز المكتبات وتحميل ملف النصوص
يلا نبدأ باستيراد الأدوات وقراءة ملف `langchain_sample.txt`:

In [1]:
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

# قراءة الملف
loader = TextLoader("langchain_sample.txt")
raw_docs = loader.load()

# تقطيع النص لقطع حجمها 500 حرف مع تداخل 50 حرف زي ما عمل المحاضر
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(raw_docs)

print(f"تم قراءة الملف وتقطيعه بنجاح إلى {len(docs)} قطعة نصية!")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_9292\2423774128.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


تم قراءة الملف وتقطيعه بنجاح إلى 6 قطعة نصية!


### 2. التخزين في FAISS والاسترجاع الأولي السريع
هنخزن القطع دي في فهرس FAISS محلي مجاني ونسترجع أفضل 6 وثائق:

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# تضمين محلي سريع
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(docs, embeddings)

# مسترجع يستخرج 6 وثائق
retriever = vector_store.as_retriever(search_kwargs={"k": 6})

query = "How can I use LangChain to build an application with memory and tools?"
retrieve_docs = retriever.invoke(query)

print(f"السؤال: {query}\n")
print(f"الترتيب الأولي السريع من FAISS (قبل الـ Reranking):")
for i, d in enumerate(retrieve_docs, start=1):
    print(f"المركز {i}: {d.page_content[:80]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

السؤال: How can I use LangChain to build an application with memory and tools?

الترتيب الأولي السريع من FAISS (قبل الـ Reranking):
المركز 1: LangChain is an open-source development framework designed to simplify the creat...
المركز 2: To build an application with memory and tools, developers instantiate a chat mod...
المركز 3: When building conversational applications, memory management is essential. LangC...
المركز 4: For interactive and dynamic problem solving, LangChain supports comprehensive to...
المركز 5: One of the fundamental pillars of LangChain is its component architecture. Model...
المركز 6: In retrieval-augmented generation (RAG) pipelines, LangChain enables retrieval w...


### 3. تجهيز نصوص الوثائق المرقمة وبرومبت الـ Reranker
دلوقتي هنجهز الوثائق في شكل قائمة مرقمة (1، 2، 3...) علشان نسأل النموذج اللغوي يرتبهم:

In [3]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# دمج الوثائق مع أرقامها
formatted_docs = "\n\n".join([f"{i}. {doc.page_content}" for i, doc in enumerate(retrieve_docs, start=1)])

# قالب التوجيه الذكي لإعادة الترتيب
template = """You are a helpful assistant.
Your task is to rank the following documents from most to least relevant to the user question.

User Question:
{question}

Documents:
{documents}

Instructions:
Think about the relevance of each document to the user question.
Return a list of document indices in ranked order starting from the most relevant.
Output format: comma-separated document indices (e.g., 2, 1, 4, 3, 5, 6)."""

prompt = ChatPromptTemplate.from_template(template)
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)

chain = prompt | llm | StrOutputParser()
print("سلسلة الـ Reranker جاهزة للتشغيل!")

سلسلة الـ Reranker جاهزة للتشغيل!


### 4. تشغيل الـ Reranker واستخراج الترتيب الجديد
تعال نشوف النموذج رتب الأرقام إزاي:

In [4]:
response = chain.invoke({"question": query, "documents": formatted_docs})
print(f"رد النموذج (ترتيب الأرقام): {response.strip()}\n")

# تحويل الأرقام لفهارس بايثون (طرح 1 لأن بايثون بيبدأ من 0)
indices = [int(x.strip()) - 1 for x in response.split(",") if x.strip().isdigit()]
print(f"الفهارس البرمجية بعد الطرح: {indices}")

# إعادة ترتيب الوثائق حسب رأي الـ LLM
ranked_docs = [retrieve_docs[i] for i in indices if 0 <= i < len(retrieve_docs)]
print(f"تمت إعادة ترتيب {len(ranked_docs)} مستند بنجاح!")

رد النموذج (ترتيب الأرقام): 2, 3, 4, 1, 5, 6

الفهارس البرمجية بعد الطرح: [1, 2, 3, 0, 4, 5]
تمت إعادة ترتيب 6 مستند بنجاح!


### 5. عرض النتائج بعد إعادة الترتيب
تعال نشوف مين المستند اللي صعد للمركز الأول دلوقتي:

In [5]:
print("=== الترتيب النهائي للوثائق بعد الـ Reranking ===\n")
for rank, doc in enumerate(ranked_docs, start=1):
    print(f"المركز {rank}:\n{doc.page_content}\n{'-'*40}")

=== الترتيب النهائي للوثائق بعد الـ Reranking ===

المركز 1:
To build an application with memory and tools, developers instantiate a chat model, equip it with appropriate tool definitions, and wrap it with an agent executor or LangGraph state machine. Conversational memory is injected into the prompt as message history, allowing the agent to recall prior tool outputs and user preferences while executing multi-step reasoning tasks. This composability makes LangChain a primary framework for developing scalable GenAI systems.
----------------------------------------
المركز 2:
When building conversational applications, memory management is essential. LangChain supports diverse memory abstractions such as ConversationBufferMemory, ChatMessageHistory, and summary memory. These components enable LLMs to maintain state across conversational turns, resolving anaphoric references and past dialogue context seamlessly in production chatbots and virtual assistants.
---------------------------------

### 6. جرب بنفسك!
جرب تغير السؤال فوق (مثلاً: اسأل عن فهارس المتجهات FAISS و Chroma) وشوف إزاي الترتيب بيتغير والمستند الخاص بالفيكتورز بيصعد للمركز الأول فوراً!